# 2. Deep learning on the per-patient time-frequency signal

The final model and how each piece contributes.

**Architecture** — two streams joined at the head:

* `Spectrum1DCNN` over the log-binned **multitaper** spectrum (16 bins, 3-15 Hz)
* `TrajectoryEncoder`, a dilated TCN over the **instantaneous-frequency
  trajectory**, which carries temporal dynamics the averaged spectrum cannot
* soft-voted with a `ResidualTCN` on the spectrum

plus 10 descriptors, 4 bilateral-asymmetry features, and validation-tuned class
priors.

Measured: **precN 0.639, precPD 0.655, precET 0.685, macro precision 0.660**,
paired +0.041 [+0.014, +0.067] over the welch baseline on 20 splits.


In [ ]:
import sys; sys.path.insert(0, '.')
import torch; torch.set_num_threads(1)
import numpy as np
from tfbench.final_model import build, evaluate, SPLITS, TL
from tfbench.cohort_strategies import NBIN

## Assemble the merged cohort

2015 + NewData + PADS, PADS capped at 90/class (measured optimum), asymmetry
carried as a missing modality because 2015 is single-limb.

In [ ]:
d = build()
y, key, SPEC = d['y'], d['key'], d['SPEC']
print(f"n={len(y)}  N={(y==0).sum()}  PD={(y==1).sum()}  ET={(y==2).sum()}")
print('transforms available:', list(SPEC))
D_desc = np.hstack([d['DESC'], d['ASYM'], d['HAVE']])
TR = d['TRAJ']
print('spectrum', SPEC['multitaper'].shape, ' descriptors+asym', D_desc.shape,
      ' trajectory', TR.shape)

## Baseline, then each component added

In [ ]:
hdr = (f"{'config':>40}{'precN':>9}{'precPD':>9}{'precET':>9}"
       f"{'macroP':>9}{'macroF1':>9}  |{'  sd':>7}")
print(hdr)
base = evaluate('welch + desc + asym (baseline)', SPEC['welch'], D_desc, None, y, key)
traj = evaluate('+ IF trajectory',                SPEC['welch'], D_desc, TR,   y, key)
mt   = evaluate('multitaper (no trajectory)',     SPEC['multitaper'], D_desc, None, y, key)
best = evaluate('FINAL: multitaper + trajectory', SPEC['multitaper'], D_desc, TR, y, key)

## Paired comparison — the only way these differences are readable

In [ ]:
def paired(a, b, name):
    diff = a - b
    print(f'  {name}:')
    for i, nm in enumerate(('precN','precPD','precET','macroP','macroF1')):
        boot = [np.mean(np.random.default_rng(s).choice(diff[:, i], len(diff),
                replace=True)) for s in range(4000)]
        lo, hi = np.percentile(boot, [2.5, 97.5])
        print(f'    {nm:>8} {diff[:, i].mean():+.3f}  [{lo:+.3f}, {hi:+.3f}]'
              f'{"  *" if lo > 0 or hi < 0 else ""}')

print(f'paired vs baseline, {SPLITS} splits:')
paired(traj, base, '+ IF trajectory')
paired(mt,   base, 'multitaper alone')
paired(best, base, 'FINAL multitaper + trajectory')

## Precision at reduced coverage

Every number above is at 100 % coverage — the model must label every patient.
Allowing it to abstain trades coverage for precision.

In [ ]:
from tfbench.selective import report
# pooled out-of-fold probabilities would be needed for a full curve; see
# reports/precision_ceiling.md for the measured version
print(open('reports/precision_ceiling.md').read()[:1800])

## Findings (`reports/final_model.md`)

* **The two gains stack super-additively**: trajectory +0.017, multitaper +0.009,
  together **+0.041** macro precision. Multitaper improves the spectral
  *estimate*; the trajectory adds temporal dynamics the spectrum cannot express.
* **Adding more features hurts.** Stability features on top drop macro precision
  0.660 -> 0.639. Seven feature unions in this project have underperformed their
  best member — at 404 patients with 49 ET, dimensionality binds harder than
  information.
* **Macro precision >0.90 is not reachable** on three classes at any coverage
  (`reports/precision_ceiling.md`). Closing that gap needs more ET patients or
  the PADS questionnaire, not more architecture.
